In [14]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Italy Shapefiles"

In [15]:
import warnings
warnings.filterwarnings("ignore")

In [16]:
NUTS0 = 'IT'
NUTS2 = 'Veneto'

In [17]:
YEAR = 2023
MONTH = 'May'
PERIOD = '1st'

In [18]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))
scaler = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Scaler.pkl', 'rb'))
imputer = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Imputer.pkl', 'rb'))

In [19]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,population,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,11.79324,45.35865,2023-05-01,veneto,abano terme,1,5,18,2023,0.201299,0.97953,0.5,-0.866025,0.845596,-0.533823,"19,349",46.86670,0.411526,0.124596,-0.361833,-0.124596,0.373138,0.096255,-0.340012,-0.096255,0.100676,0.062158,0.071315,0.062158,18.58,26.590,10.570,7.626000,0.600000,8.080000,-0.791538,18.365714,6.326364,21.500769,6.950000,75.863498,129.176716,381.764019,8982.126981,2074.539971,3,170.409233,10.722186,179.966181,0.0,1.063084,31,98.0,36,98.0,30,98.0,12,12,1,6,7,2,0,228,0,0
1,12.04199,45.06525,2023-05-01,veneto,adria,1,5,18,2023,0.201299,0.97953,0.5,-0.866025,0.845596,-0.533823,"20,233",46.64640,0.296728,0.041635,-0.291255,-0.041635,0.387057,0.130002,-0.367259,-0.130002,0.141819,0.138328,0.109765,0.138328,16.54,25.542,7.538,6.377211,0.808571,8.476395,-1.590923,16.728553,4.271543,21.836459,5.637644,88.213701,203.268859,718.387449,6701.380530,1644.125334,2,145.603087,-1.354496,180.070371,0.0,1.206604,31,99.0,36,99.0,30,99.0,12,12,1,6,7,2,0,57,0,0
2,10.77673,45.55680,2023-05-01,veneto,affi,1,5,18,2023,0.201299,0.97953,0.5,-0.866025,0.845596,-0.533823,"2,297",46.81410,0.354822,0.095025,-0.353391,-0.095025,0.456411,0.125488,-0.417188,-0.125488,0.121709,0.068839,0.090199,0.068839,15.19,19.170,11.210,5.821667,0.738000,8.631538,0.720667,17.207647,3.983750,18.918421,6.878889,29.987989,65.839366,178.487671,5306.025387,738.794180,6,207.641691,201.821006,178.733735,0.0,1.815834,31,84.0,30,84.0,30,84.0,10,10,1,6,6,2,0,78,0,0
3,11.96539,45.17531,2023-05-01,veneto,agna,1,5,18,2023,0.201299,0.97953,0.5,-0.866025,0.845596,-0.533823,"3,400",46.73306,0.291590,-0.007849,-0.311519,0.007849,0.372874,0.089191,-0.373604,-0.089191,0.108124,0.099352,0.078616,0.099352,17.94,24.010,11.870,4.870000,0.610000,7.752222,-2.034615,15.791176,4.124000,21.578571,5.119231,181.904724,304.444748,641.894761,16631.850238,1165.197945,3,121.193871,0.122498,180.188821,0.0,3.769479,31,99.0,36,99.0,30,99.0,12,12,1,6,7,2,0,20,0,0
4,12.04755,46.30297,2023-05-01,veneto,agordo,1,5,18,2023,0.201299,0.97953,0.5,-0.866025,0.845596,-0.533823,"4,249",47.84463,0.317522,-0.165724,-0.385563,0.165724,0.384072,-0.044801,-0.399386,0.044801,0.045556,0.057840,0.025550,0.057840,8.68,11.890,5.470,1.921429,-4.077692,5.623333,-2.298000,9.285385,-0.998333,11.544545,0.444545,27.414888,90.526653,546.465380,15023.566702,684.464446,18,124.001040,1470.386475,152.701532,0.0,2.289867,15,91.0,10,97.0,10,97.0,5,5,7,1,1,2,0,29,0,0


In [20]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

X_test = scaler.transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)


In [21]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,11.73897,44.96915,polesella,1,5,2023,1.616206e-04
1,12.62107,45.75752,motta da livenza,1,5,2023,1.136032e-04
2,12.64550,45.53937,jesolo,1,5,2023,6.794780e-05
3,12.53799,45.60103,musile da piave,1,5,2023,5.968710e-05
4,12.39607,45.79141,saint polo da piave,1,5,2023,4.931781e-05
...,...,...,...,...,...,...,...
566,11.71465,46.07125,lamon,1,5,2023,2.289352e-12
567,12.05493,46.45349,selva da cadore,1,5,2023,1.971717e-12
568,11.85565,46.37360,falcade,1,5,2023,1.444506e-12
569,11.88464,46.32739,canale da agordo,1,5,2023,1.440810e-12


In [22]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

FileNotFoundError: [Errno 2] No such file or directory: '../../data/Veneto/IT_Veneto_Bins_2023.csv'

In [22]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

In [23]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}_(new).csv", encoding = enc, index = False)

In [24]:
##TODO Visualisation of results

,municipality,geometry,x,y,day,month,year,probability
0,chioggia,"POLYGON ((12.29589 45.33225, 12.29961 45.30769...",12.24756,45.27153,1,5,2023,0.003282
1,venezia,"POLYGON ((12.58835 45.53969, 12.58795 45.53951...",12.32478,45.43497,1,5,2023,0.002148
2,codevigo,"POLYGON ((12.12752 45.30091, 12.12959 45.30046...",12.18085,45.26512,1,5,2023,0.001883
3,rosolina,"POLYGON ((12.32883 45.14614, 12.32880 45.14586...",12.30160,45.08601,1,5,2023,0.000718
4,cavallino treporti,"POLYGON ((12.51153 45.50279, 12.51263 45.50274...",12.49633,45.47024,1,5,2023,0.000645
